In [ ]:
#Install pyspark if not already installed, delete the '#' symbol
#pip install pyspark

In [1]:
#Import required libraries
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, desc, expr, row_number, sum as _sum
from pyspark.sql.window import Window

# Initialise spark
    - In this step, we initialize the Spark environment and configure it to communicate with our manually installed local Cassandra instance via the `spark-cassandra-connector`.

In [12]:
#Specify environment versions for reproducibility
#Print python version
print(f"Python Version: {sys.version}")

#Need to include cassandra package
#This also includes the cassandra version (.config)
spark = SparkSession.builder \
    .appName("MovieLensSparkCassandraPipeline") \
    .config("spark.cassandra.connection.host", "127.0.0.1") \
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.2.0") \
    .getOrCreate()

sc = spark.sparkContext
#Print spark version
print(f"Apache Spark Version: {spark.version}")

Python Version: 3.6.8 (default, Nov 14 2023, 16:29:52) 
[GCC 4.8.5 20150623 (Red Hat 4.8.5-44)]


/usr/local/lib/python3.6/site-packages/pyspark/context.py:238: FutureWarning: Python 3.6 support is deprecated in Spark 3.2.
  FutureWarning


Apache Spark Version: 3.2.4


# Load and parse dataset into RDD
    - To ensure stability and bypass Hadoop NameNode port limitations, we load the raw `u.data`, `u.user`, and `u.item` files directly from the local Linux filesystem (`/tmp/ml-100k/`). The data is parsed from delimited strings into strongly typed tuples.

In [3]:
#HDFS keeps crashing, bypass by installing to temp
LOCAL_BASE = "file:///tmp/ml-100k/"

raw_data_rdd = sc.textFile(LOCAL_BASE + "u.data")
raw_user_rdd = sc.textFile(LOCAL_BASE + "u.user")
raw_item_rdd = sc.textFile(LOCAL_BASE + "u.item")

#Parse RDD elements mapping data structures and casting types safely
data_parsed_rdd = raw_data_rdd.map(lambda line: line.split('\t')) \
                              .map(lambda x: (int(x[0]), int(x[1]), float(x[2]), int(x[3])))

user_parsed_rdd = raw_user_rdd.map(lambda line: line.split('|')) \
                              .map(lambda x: (int(x[0]), int(x[1]), x[2], x[3], x[4]))

def parse_item(line):
    parts = line.split('|')
    movie_id = int(parts[0])
    title = parts[1]
    genres = [int(g) for g in parts[5:]]  # 19 binary genre columns
    return (movie_id, title) + tuple(genres)

item_parsed_rdd = raw_item_rdd.map(parse_item)

# Transform RDD to spark dataframe and clean
    - We convert the RDDs into Spark DataFrames, apply schema definitions, and clean the data by dropping duplicates and null values. Finally, we register Temporary SQL Views to allow for native Spark SQL queries.

In [4]:
#Define genres
genre_names = ["unknown", "Action", "Adventure", "Animation", "Childrens", "Comedy", 
               "Crime", "Documentary", "Drama", "Fantasy", "Film_Noir", "Horror", 
               "Musical", "Mystery", "Romance", "Sci_Fi", "Thriller", "War", "Western"]

#Create dataframe for each dataset: user, item, data
df_data = spark.createDataFrame(data_parsed_rdd, ["user_id", "movie_id", "rating", "timestamp"])
df_user = spark.createDataFrame(user_parsed_rdd, ["user_id", "age", "gender", "occupation", "zip_code"])
df_item = spark.createDataFrame(item_parsed_rdd, ["movie_id", "movie_title"] + genre_names)

#Data cleaning: Drop duplicates or null entries if any exist
df_data = df_data.dropDuplicates().dropna()
df_user = df_user.dropDuplicates().dropna()
df_item = df_item.dropDuplicates().dropna()

#Register temporary views for Spark SQL execution 
df_data.createOrReplaceTempView("view_data")
df_user.createOrReplaceTempView("view_user")
df_item.createOrReplaceTempView("view_item")

# Execute analysis
    - Here we perform our core data analysis tasks:
        Tasks i & ii: Calculate the Top 10 movies by average rating.
        Task iii: Identify the favourite movie genre for users who have submitted 50 or more ratings (using Window functions).
        Task iv: Filter users strictly under 20 years old.
        Task v: Filter users who are scientists aged between 30 and 40.

In [5]:
#Print statement for each task for easier visualisation of results
#i for u.item, d for u.data

print("\n--- Task i & ii: Top 10 Movies by Average Rating ---") 
#Query checks the average rating and pulls titles 
avg_ratings_df = spark.sql("""
    SELECT i.movie_id, i.movie_title, ROUND(AVG(d.rating), 2) as avg_rating
    FROM view_data d
    JOIN view_item i ON d.movie_id = i.movie_id
    GROUP BY i.movie_id, i.movie_title
""")
top_10_movies = avg_ratings_df.orderBy(desc("avg_rating")).limit(10) 
top_10_movies.show(truncate=False) 

#For task iii, we need to do some pre-processing
print("\n--- Task iii: Users (>=50 ratings) & Their Favourite Genre ---") 
#Step 1: Filter users with >= 50 ratings 
frequent_raters = df_data.groupBy("user_id").count().filter("count >= 50")

#Step 2: Melt genre flags into rows to easily aggregate frequencies 
user_movies_genres = df_data.join(frequent_raters, "user_id").join(df_item, "movie_id")
genre_sums = [_sum(col(g)).alias(g) for g in genre_names]
user_genre_matrix = user_movies_genres.groupBy("user_id").agg(*genre_sums)

stack_expr = ", ".join([f"'{g}', `{g}`" for g in genre_names])
melted_genres_df = user_genre_matrix.select("user_id", expr(f"stack({len(genre_names)}, {stack_expr}) as (favourite_genre, rating_count)"))

#Step 3: Pinpoint the top genre via Window Row Numbering
window_spec = Window.partitionBy("user_id").orderBy(desc("rating_count"))
fav_genre_df = melted_genres_df.withColumn("rn", row_number().over(window_spec)) \
                               .filter("rn = 1").drop("rn")
fav_genre_df.limit(10).show() 

#We get data from u.user data, select all where age is less than 20
print("\n--- Task iv: Users less than 20 years old ---") 
users_under_20 = spark.sql("SELECT * FROM view_user WHERE age < 20")
users_under_20.limit(10).show() 

#Select all sccientist between 30 and 40 years old. use LOWER to standardise for easy query. no messy formatting
print("\n--- Task v: Scientists between 30 and 40 years old ---") 
scientists_30_40 = spark.sql("""
    SELECT * FROM view_user 
    WHERE LOWER(occupation) = 'scientist' AND age BETWEEN 30 AND 40
""")

#Show only 10 entries
scientists_30_40.limit(10).show()


--- Task i & ii: Top 10 Movies by Average Rating ---
+--------+-------------------------------------------------+----------+
|movie_id|movie_title                                      |avg_rating|
+--------+-------------------------------------------------+----------+
|1122    |They Made Me a Criminal (1939)                   |5.0       |
|1500    |Santa with Muscles (1996)                        |5.0       |
|1599    |Someone Else's America (1995)                    |5.0       |
|1467    |Saint of Fort Washington, The (1993)             |5.0       |
|1293    |Star Kid (1997)                                  |5.0       |
|1536    |Aiqing wansui (1994)                             |5.0       |
|814     |Great Day in Harlem, A (1994)                    |5.0       |
|1189    |Prefontaine (1997)                               |5.0       |
|1653    |Entertaining Angels: The Dorothy Day Story (1996)|5.0       |
|1201    |Marlene Dietrich: Shadow and Light (1996)        |5.0       |
+--------+

# Write to cassandra

In [6]:
#Our tables in cassandra are: movie_avg_ratings, user_favourite_genres, filtered_users (for task iv and v)
def write_to_cassandra(df, table_name):
    df.write \
      .format("org.apache.spark.sql.cassandra") \
      .options(table=table_name, keyspace="movielens_analysis") \ #Our keyspace in cassandra is this
      .mode("append") \
      .save()

#Write the average ratings, favourite genres, and filtered users to the tables in cassandra
print("\nWriting to Cassandra...")
write_to_cassandra(avg_ratings_df, "movie_avg_ratings")
write_to_cassandra(fav_genre_df, "user_favourite_genres")

#Union target groups iv and v to seamlessly stream into a uniform users table
combined_users = users_under_20.unionByName(scientists_30_40, allowMissingColumns=True).dropDuplicates()
write_to_cassandra(combined_users, "filtered_users")

#Print statement to know we successfully exported to cassandra
print("[SUCCESS] Pipeline successfully exported structured records into Cassandra.")


Writing to Cassandra...
[SUCCESS] Pipeline successfully exported structured records into Cassandra.


# Read back from cassandra

In [13]:
print("\n=== VALIDATION FROM CASSANDRA ===")

#Function to get back data from cassandra
def validate_table(table_name):
    return spark.read \
                .format("org.apache.spark.sql.cassandra") \
                .options(table=table_name, keyspace="movielens_analysis") \
                .load()

#Use the function to get the data; show top 5 highest rated movies
print("Validating 'movie_avg_ratings' Dataframe:")
validate_table("movie_avg_ratings").orderBy(desc("avg_rating")).show(5)

#Show 5 entries of user's favourite data
print("Validating 'user_favourite_genres' Dataframe:")
validate_table("user_favourite_genres").orderBy(desc("rating_count")).show(5)

#Show 5 entries of the filtered users
print("Validating 'filtered_users' Dataframe:")
validate_table("filtered_users").show(5)

#Safely close the environment
spark.stop()


=== VALIDATION FROM CASSANDRA ===
Validating 'movie_avg_ratings' Dataframe:
+--------+----------+--------------------+
|movie_id|avg_rating|         movie_title|
+--------+----------+--------------------+
|    1599|       5.0|Someone Else's Am...|
|    1653|       5.0|Entertaining Ange...|
|    1189|       5.0|  Prefontaine (1997)|
|    1467|       5.0|Saint of Fort Was...|
|    1536|       5.0|Aiqing wansui (1994)|
+--------+----------+--------------------+
only showing top 5 rows

Validating 'user_favourite_genres' Dataframe:
+-------+---------------+------------+
|user_id|favourite_genre|rating_count|
+-------+---------------+------------+
|    655|          Drama|         410|
|    405|          Drama|         309|
|    537|          Drama|         251|
|    450|          Drama|         237|
|     13|          Drama|         218|
+-------+---------------+------------+
only showing top 5 rows

Validating 'filtered_users' Dataframe:
+-------+---+------+----------+--------+
|user_id|